# CSV Normalizer

Converts raw calculation CSV files (`source/`) into **streamlined normalized CSVs**
(`normalized/`) with 3 machine-readable columns:

| Column | Source | Notes |
|--------|--------|-------|
| `group` | `Group` | Forward-filled |
| `indicator` | `Indicator` | Unchanged |
| `calculation` | `Calculation Comprehensive monitoring` | IDs → `question_name` + formula normalization |

**Dropped columns**: `#`, `Used questions`, `Feasible?`, `Used questionnaire(s)`, `Guidance`,
`Logic`, `Remarks`, `Verify with Ima / DWS / Tech`, `checked Lotte`.

> **Why no `used_questions` column?** Every `question_name` a row depends on is already
> embedded in its `calculation` formula. A separate hand-curated `used_questions` column
> was redundant and error-prone — when edited manually a name could be omitted, silently
> dropping it from config generation. `calculation` is now the single source of truth;
> the downstream `visualization_config.ipynb` parses `question_name` tokens directly from it.

> **Target format — VizCalc.** The `calculation` column should be authored in **VizCalc**,
> the Excel-style grammar in [`README.md`](README.md): inputs as `[question_name]`, one
> outermost `UPPERCASE(...)` output function, ASCII operators (`<>` = not-equal), and
> `AND/OR/NOT/IF` for logic. This normalizer resolves 13-digit IDs (including bracketed
> `[1749…]`) to `question_name` and maps Unicode operators to their ASCII/VizCalc form;
> VizCalc formulas otherwise pass through unchanged. Legacy free-prose rows still work —
> they fall through to the heuristic builder downstream — but only VizCalc guarantees that
> every referenced question is captured (see the README "before → after" note).

Only **feasible rows** (`Feasible? == Yes`) are written. All rows in the output are
considered valid — no feasibility filter is needed downstream.

## Workflow

```
source/*.csv  →  normalize_csv.ipynb  →  normalized/*.csv  →  visualization_config.ipynb  →  output/*.json
```

**Run from** `scripts/visualization-config/`.

In [1]:
import json
import re
import csv
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
FORMS_DIR = REPO_ROOT / "backend/source/forms"
SOURCE_DIR = Path("source")
NORMALIZED_DIR = Path("normalized")

NORMALIZED_DIR.mkdir(exist_ok=True)

assert FORMS_DIR.exists(), f"Forms dir not found: {FORMS_DIR}"
assert SOURCE_DIR.exists(), f"Source dir not found: {SOURCE_DIR}"

print(f"REPO_ROOT     : {REPO_ROOT}")
print(f"FORMS_DIR     : {FORMS_DIR}")
print(f"SOURCE_DIR    : {SOURCE_DIR}")
print(f"NORMALIZED_DIR: {NORMALIZED_DIR}")

REPO_ROOT     : /home/iwan/Akvo/iwsims
FORMS_DIR     : /home/iwan/Akvo/iwsims/backend/source/forms
SOURCE_DIR    : source
NORMALIZED_DIR: normalized


In [2]:
# Build lookup maps from form source JSONs

ID_RE = re.compile(r'\b\d{13}\b')

qid_map: dict = {}   # question_id (int) -> {name, label, type, form_id}
fid_map: dict = {}   # form_id (int) -> form_name (str)


def _index_form(form_data: dict) -> None:
    form_id = form_data["id"]
    form_name = form_data.get("form") or form_data.get("name", str(form_id))
    fid_map[form_id] = form_name
    for qg in form_data.get("question_groups", []):
        for q in qg.get("questions", []):
            qid_map[q["id"]] = {
                "name": q.get("name", ""),
                "label": q.get("label", ""),
                "type": q.get("type", ""),
                "form_id": form_id,
                "form_name": form_name,
            }


for f in sorted(FORMS_DIR.glob("*.json")):
    if "example" in f.name:
        continue
    try:
        _index_form(json.loads(f.read_text()))
    except Exception as e:
        print(f"Warning: {f.name}: {e}")

print(f"Indexed {len(qid_map)} questions across {len(fid_map)} forms.")
print("\nForms:")
for fid, fname in sorted(fid_map.items()):
    print(f"  {fid}  {fname}")

Indexed 1095 questions across 18 forms.

Forms:
  1748903240763  WAF Wastewater Treatment Plant
  1748905550055  WAF Wastewater Treatment Plant - Monitoring
  1748918946591  WAF Wastewater Treatment Plant - Quick Monitoring
  1749611049520  Wastewater Pump Station
  1749611905372  Wastewater Pump Station - Monitoring
  1749621221728  Rural Water Project
  1749621962296  Rural Water Project - Monitoring
  1749623934933  EPS Inspection
  1749624452908  EPS Projects Construction - Monitoring
  1749627302948  Wastewater Pump Station - Quick Monitoring
  1749631041125  Rural Water Project - Quick Monitoring
  1749632545233  EPS Water Quality Testing - Monitoring
  1749634736797  WAF Water Treatment Plant
  1749640508297  WAF Water Treatment Plant - Quick Monitoring
  1749652214711  WAF Water Treatment Plant - Monitoring
  16993539153551  Short HH
  16993539153552  Short HH Monitoring
  17993639153662  Short HH Testimonials


In [3]:
# Replacement helpers

def replace_qids(text: str) -> str:
    """Replace each 13-digit question ID with its question_name.

    IDs that cannot be resolved are left as-is with a '??' suffix. Bracketed VizCalc
    inputs like [1900000000101] resolve too — '[' and ']' are word boundaries, so a
    raw-ID formula and a VizCalc formula both normalize to [question_name] form.
    """
    def _sub(m) -> str:
        qid = int(m.group())
        info = qid_map.get(qid, {})
        name = info.get("name")
        return name if name else f"{m.group()}??"

    return ID_RE.sub(_sub, text)


def to_machine_form(text: str) -> str:
    """Convert human-written formula to machine-readable single-line form.

    Steps:
    1. Replace 13-digit IDs with question_names
    2. Normalize Unicode operators to ASCII / VizCalc: ≥ >= ≤ <= ≠ <> × *
    3. Flatten bullet-point conditions into AND(...) notation
    4. Collapse multi-line text into a single line separated by ' | '

    Formulas already authored in VizCalc (README.md) pass through unchanged: a
    single-line `OUTPUT([question_name],...)` has no bullets and no Unicode, so only
    bracketed IDs (if any) get resolved to question_names. `≠` maps to VizCalc `<>`
    (not `!=`), matching the grammar's not-equal operator.
    """
    if not text.strip():
        return ""

    # 1. Replace IDs with question_names
    text = replace_qids(text)

    # 2. Normalize Unicode operators (≠ -> VizCalc <>)
    text = (
        text.replace("≥", ">=").replace("≤", "<=")
            .replace("≠", "<>").replace("×", "*")
            .replace("−", "-")
    )

    # 3. Process line by line: collapse bullet points into AND() groups
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    parts = []
    bullet_conds = []

    for line in lines:
        is_bullet = bool(re.match(r'^[•·\*\-]\s+', line))
        cleaned = re.sub(r'^[•·\*\-]\s+', '', line).rstrip(",")

        if is_bullet:
            # Strip trailing AND/OR connective — it's implicit
            cond = re.sub(r',?\s*(AND|OR)\s*$', '', cleaned, flags=re.IGNORECASE).strip()
            if cond:
                bullet_conds.append(cond)
        else:
            if bullet_conds:
                parts.append("AND(" + "; ".join(bullet_conds) + ")")
                bullet_conds = []
            if cleaned:
                parts.append(cleaned)

    if bullet_conds:
        parts.append("AND(" + "; ".join(bullet_conds) + ")")

    return " | ".join(parts)


# Sanity checks
tests = [
    # Bullet list with IDs
    "Plant is OPERATIONAL when :\n  • 1749700156602 = No (no constraints), AND\n  • 1749652417794 > 0, AND\n  • 1754995400013 = No",
    # Unicode operator
    "COUNT DISTINCT IDs WHERE 1900000000101 ≥ 12 months.",
    # Simple formula
    "COUNT DISTINCT Registration records by 1749635249444.",
    # VizCalc authored with a bracketed raw ID -> resolves, passes through unchanged
    "RECENT([1900000000101],12)",
    # VizCalc with Unicode not-equal -> ASCII <>
    "COUNT([1749635249444]≠\"\")",
]
for t in tests:
    print(f"IN : {t[:80]!r}")
    print(f"OUT: {to_machine_form(t)!r}")
    print()

IN : 'Plant is OPERATIONAL when :\n  • 1749700156602 = No (no constraints), AND\n  • 174'
OUT: 'Plant is OPERATIONAL when : | AND(has_production_constraints = No (no constraints); daily_production_megalitres > 0; pumps_rising_main_has_risks = No)'

IN : 'COUNT DISTINCT IDs WHERE 1900000000101 ≥ 12 months.'
OUT: 'COUNT DISTINCT IDs WHERE date_of_inspection >= 12 months.'

IN : 'COUNT DISTINCT Registration records by 1749635249444.'
OUT: 'COUNT DISTINCT Registration records by plant_name.'

IN : 'RECENT([1900000000101],12)'
OUT: 'RECENT([date_of_inspection],12)'

IN : 'COUNT([1749635249444]≠"")'
OUT: 'COUNT([plant_name]<>"")'



In [4]:
# Output column schema

OUTPUT_FIELDS = ["group", "indicator", "calculation"]

# Exact source column names (vary slightly between CSVs for the calculation col).
# question_names are resolved from the calculation formula itself — no separate
# hand-curated "used_questions" column is emitted (calculation is the source of truth).
SOURCE_FIELD_MAP = {
    "group":       lambda r: r.get("Group", ""),
    "indicator":   lambda r: r.get("Indicator", "").strip(),
    "calculation": lambda r: to_machine_form(r.get("Calculation Comprehensive monitoring", "")),
}

print("Output columns:", OUTPUT_FIELDS)
print("Feasible rows only: Yes")
print("Formula normalization: to_machine_form()")

Output columns: ['group', 'indicator', 'calculation']
Feasible rows only: Yes
Formula normalization: to_machine_form()


In [5]:
# Normalize all CSVs

def normalize_csv(source_path: Path, out_path: Path) -> dict:
    """Read source CSV, keep feasible rows, output streamlined normalized CSV.

    Returns stats: {total_src, feasible, unresolved_ids}.
    """
    with open(source_path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    # Forward-fill Group (some rows leave it blank, inheriting from above)
    current_group = ""
    for r in rows:
        g = r.get("Group", "").strip()
        if g:
            current_group = g
        else:
            r["Group"] = current_group

    # Keep only feasible rows
    feasible_rows = [r for r in rows if r.get("Feasible?", "").strip() == "Yes"]

    unresolved = set()
    out_rows = []

    for r in feasible_rows:
        out_row = {field: fn(r) for field, fn in SOURCE_FIELD_MAP.items()}

        # Collect any unresolved IDs (marked with ?? suffix by replace_qids)
        for bad in re.findall(r'\d{13}\?\?', out_row["calculation"]):
            unresolved.add(bad.replace("??", ""))

        out_rows.append(out_row)

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS)
        writer.writeheader()
        writer.writerows(out_rows)

    return {
        "total_src": len(rows),
        "feasible": len(out_rows),
        "unresolved": sorted(unresolved),
    }


for csv_file in sorted(SOURCE_DIR.glob("*.csv")):
    out_path = NORMALIZED_DIR / csv_file.name
    stats = normalize_csv(csv_file, out_path)
    print(f"\n{csv_file.name}")
    print(f"  Source rows : {stats['total_src']}")
    print(f"  Feasible    : {stats['feasible']}")
    print(f"  Written to  : {out_path}")
    if stats["unresolved"]:
        print(f"  !! Unresolved IDs: {stats['unresolved']}")
    else:
        print("  All IDs resolved.")


1748903240763.csv
  Source rows : 52
  Feasible    : 38
  Written to  : normalized/1748903240763.csv
  All IDs resolved.

1749611049520.csv
  Source rows : 34
  Feasible    : 24
  Written to  : normalized/1749611049520.csv
  All IDs resolved.

1749634736797.csv
  Source rows : 44
  Feasible    : 36
  Written to  : normalized/1749634736797.csv
  All IDs resolved.


In [6]:
# Spot-check: show sample rows from each normalized CSV

for csv_file in sorted(NORMALIZED_DIR.glob("*.csv")):
    with open(csv_file, newline="") as f:
        rows = list(csv.DictReader(f))

    print(f"\n{'='*70}")
    print(f"{csv_file.name}  ({len(rows)} rows)")
    print(f"Columns: {list(rows[0].keys()) if rows else '—'}")
    print()

    for r in rows[:5]:
        print(f"  [{r['indicator']}]")
        calc = r['calculation']
        print(f"    calculation: {calc[:110]!r}")
        print()


1748903240763.csv  (38 rows)
Columns: ['group', 'indicator', 'calculation']

  [Total WWTPs]
    calculation: 'COUNTDISTINCT(COALESCE([plant_name],[geolocation]))'

  [Inspected (12 mo)]
    calculation: 'RECENT([inspection_date],12)'

  [Effluent Compliance]
    calculation: 'COMPLIANT(AND([bod]<40,[chemical_oxygen_demand]<100,[total_dissolved_solids]<1000))'

  [Operationality]
    calculation: 'MAP([geolocation])'

  [Effluent compliance]
    calculation: 'MAP([bod])'


1749611049520.csv  (24 rows)
Columns: ['group', 'indicator', 'calculation']

  [Total Pump Stations]
    calculation: 'COUNTDISTINCT(COALESCE([pump_station_name],[gps_location]))'

  [Inspected (12 mo)]
    calculation: 'RECENT([inspection_date],12)'

  [% Operational]
    calculation: 'PERCENT(OR([station_status]="Operational",AND([electrical_panel]="Satisfactory",[pump_station_lid]="Covered",['

  [Critical issues]
    calculation: 'COUNT(OR(IN([station_status],"Not Operational","Overflowing","Blocked"),AND(IN([el

In [7]:
# Question name reference index
# Lists all unique question_names referenced in the 'calculation' formulas across
# normalized CSVs. Tokens are matched against the form index (qid_map names) so that
# ordinary English words in the formula prose are not mistaken for question_names.

VALID_QNAMES = {info["name"] for info in qid_map.values() if info.get("name")}
QNAME_RE = re.compile(r'\b([a-z][a-z0-9_]{2,}[a-z0-9])\b')

for csv_file in sorted(NORMALIZED_DIR.glob("*.csv")):
    with open(csv_file, newline="") as f:
        rows = list(csv.DictReader(f))

    name_to_indicators = {}
    for r in rows:
        names = [
            n for n in QNAME_RE.findall(r.get("calculation", ""))
            if '_' in n and n in VALID_QNAMES
        ]
        for name in names:
            name_to_indicators.setdefault(name, []).append(r["indicator"].strip())

    print(f"\n{'='*60}")
    print(f"{csv_file.name} — {len(name_to_indicators)} unique question_names")
    for name in sorted(name_to_indicators):
        indicators = ", ".join(name_to_indicators[name][:3])
        more = f" (+{len(name_to_indicators[name])-3} more)" if len(name_to_indicators[name]) > 3 else ""
        print(f"  {name:<50} {indicators}{more}")


1748903240763.csv — 32 unique question_names
  any_upgrading_programs                             Upgrading programs
  building_conditions                                Building condition
  can_take_sample                                    No sample taken
  chemical_oxygen_demand                             Effluent Compliance, Critical issues
  comment_on_operation_of_aerobic_lagoons            Aerobic lagoons
  comment_on_operation_of_idea_lagoons               IDEA lagoons
  comment_on_operation_of_imhoff                     Imhoff tanks
  comment_on_operation_of_oxidation_ponds            Oxidation ponds
  comment_on_operation_of_pasveer_ditches            Pasveer ditches
  comment_on_operation_of_polishing_ponds            Polishing ponds
  comment_on_operation_of_sludge_digester            Sludge digester
  comment_on_operation_of_the_sludge_aerobic_digestion Sludge aerobic digestion
  comment_on_the_drying_beds                         Drying beds
  comment_on_the_influent_pum